# Install the Package
Here we're installing it directly from GitHub while it's in development.

In [ ]:
!uv pip install "vanna[fastapi,ollama]"
!uv pip install python-dotenv
!uv pip install httpx

Using Python 3.13.3 environment at: C:\Users\koakepan\Downloads\NLP-SQL-AI\.venv
Audited 1 package in 123ms


# Download a Sample Database

In [12]:
import httpx

with open("Chinook.sqlite", "wb") as f:
    with httpx.stream("GET", "https://vanna.ai/Chinook.sqlite") as response:
        for chunk in response.iter_bytes():
            f.write(chunk)

# Imports

In [1]:
import os
import requests
from dotenv import load_dotenv # Import load_dotenv for environment variables   

# # Set up Ollama for local LLM
from vanna.integrations.ollama import OllamaLlmService

# Import SQLite tool
from vanna.tools import RunSqlTool
from vanna.integrations.sqlite import SqliteRunner

# Import agent memory tools
from vanna.tools.agent_memory import SaveQuestionToolArgsTool, SearchSavedCorrectToolUsesTool, SaveTextMemoryTool
from vanna.integrations.local.agent_memory import DemoAgentMemory
from vanna.core.registry import ToolRegistry

# Import user authentication classes
from vanna.core.user import UserResolver, User, RequestContext

# Import base classes
from vanna import Agent
from vanna.core.registry import ToolRegistry
from vanna.tools import VisualizeDataTool

# Run the server with FastAPI
from vanna.servers.fastapi import VannaFastAPIServer

In [2]:
load_dotenv(".env")  # Load environment variables from .env file

True

In [4]:
# print("Environment variables loaded.")
# print(f"AZURE_OPENAI_API_KEY: {os.getenv('AZURE_OPENAI_API_KEY')}")
# print(f"AZURE_OPENAI_DEPLOYMENT_NAME: {os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}")
# print(f"AZURE_OPENAI_ENDPOINT: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
# print(f"AZURE_OPENAI_API_VERSION: {os.getenv('AZURE_OPENAI_API_VERSION')}")
# print(f"OLLAMA_API_HOST: {os.getenv('OLLAMA_API_HOST')}")
# print(f"OLLAMA_MODEL_NAME: {os.getenv('OLLAMA_MODEL_NAME')}")

# Configure Your LLM

Chat with your SQLite database using Ollama

The model must support tools calling.

In [7]:
llm = OllamaLlmService(
    model = os.getenv("OLLAMA_MODEL_NAME"),
    host = os.getenv("OLLAMA_API_HOST")
)

# Configure Your Database Connection

In [8]:
# Set up database connection
db_tool = RunSqlTool(
    sql_runner=SqliteRunner(database_path="Chinook.sqlite")
)

# Configure Your Agent Memory  

Agent memory allows your agent to learn from past interactions by storing successful question-SQL pairs (or really more generically question-tool-argument tuples). When users ask similar questions in the future, the agent can search its memory to find relevant examples, improving accuracy and reducing the need to regenerate SQL from scratch.

In [9]:
# Set up agent memory for learning from questions and SQL
agent_memory = DemoAgentMemory(max_items=1000)

# Register memory tools (they access agent_memory via ToolContext)
tools = ToolRegistry()
tools.register_local_tool(SaveQuestionToolArgsTool(), access_groups=['admin'])
tools.register_local_tool(SearchSavedCorrectToolUsesTool(), access_groups=['admin', 'user'])
tools.register_local_tool(SaveTextMemoryTool(), access_groups=['admin', 'user'])

# Configure User Authentication

Define how users are identified and authenticated in your application. This example shows a simple cookie-based authentication system that assigns group memberships based on user email.

In [10]:
# Create a simple user resolver
class SimpleUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        user_email = request_context.get_cookie('vanna_email') or 'guest@example.com'
        group = 'admin' if user_email == 'admin@example.com' else 'user'
        return User(id=user_email, email=user_email, group_memberships=[group])

# Initialize the user resolver
user_resolver = SimpleUserResolver()

# Create Your Agent

In [11]:
# Register tools
tools = ToolRegistry()
tools.register_local_tool(db_tool, access_groups=['admin', 'user'])
tools.register_local_tool(SaveQuestionToolArgsTool(), access_groups=['admin'])
tools.register_local_tool(SearchSavedCorrectToolUsesTool(), access_groups=['admin', 'user'])
tools.register_local_tool(SaveTextMemoryTool(), access_groups=['admin', 'user'])
tools.register_local_tool(VisualizeDataTool(), access_groups=['admin', 'user'])

# Create your agent
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=user_resolver,
    agent_memory=agent_memory
)

# Run the Server

The notebook includes the Chinook sample database (a music store) where you can try natural language queries like:

1. What are the top 5 selling albums?
2. Show me total sales by country
3. Which artists have the most tracks?

In [12]:
server = VannaFastAPIServer(agent)
server.run()  # Access at http://localhost:8000

INFO:     Started server process [12924]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Your app is running at:
http://localhost:8000
INFO:     127.0.0.1:59788 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:59788 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:54341 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:54341 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK
INFO:     127.0.0.1:58318 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK


Error in send_message (conversation_id=1770181840469-g0inocw3s): invalid character '\n' in string literal (status code: -1)
Traceback (most recent call last):
  File "c:\Users\koakepan\Downloads\NLP-SQL-AI\.venv\Lib\site-packages\vanna\core\agent\agent.py", line 162, in send_message
    async for component in self._send_message(
    ...<2 lines>...
        yield component
  File "c:\Users\koakepan\Downloads\NLP-SQL-AI\.venv\Lib\site-packages\vanna\core\agent\agent.py", line 653, in _send_message
    response = await self._handle_streaming_response(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\koakepan\Downloads\NLP-SQL-AI\.venv\Lib\site-packages\vanna\core\agent\agent.py", line 1356, in _handle_streaming_response
    async for chunk in self.llm_service.stream_request(request):
    ...<5 lines>...
            accumulated_tool_calls.extend(chunk.tool_calls)
  File "c:\Users\koakepan\Downloads\NLP-SQL-AI\.venv\Lib\site-packages\vanna\integrations\

KeyboardInterrupt: 